<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Ungraded_Assignments/Ungraded_A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ungraded Assignment 3 - Computer vision - classification task
1. Use any of the below datasets to train a CNN classifier. Use Notedbook 7c as a guide (which used ResNet 18 on CIFAR-10 data)
2. Read this research artcicle that introduced GradCAM https://arxiv.org/pdf/1610.02391

You are open to use your own dataset too. Good place for innovative applications (Computer Vision + FFNN  are first steps for many ideas)


---

# Datasets
### Agriculture and food security
| Dataset | Task | Size | License |
|---|---|---|---|
| [PlantVillage](https://github.com/spMohanty/PlantVillage-Dataset) | crop disease: 14 crops, 26 diseases | ~54,300 leaf images | see dataset card |
| [DeepWeeds](https://zenodo.org/record/7939059) | 8 weed species + negative | 17,509 field images | CC BY 4.0 (images) |
| [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) | 101 dishes | 101,000 images | research use; `torchvision.datasets.Food101` |

### Remote sensing, transport and health
| Dataset | Task | Size | License |
|---|---|---|---|
| [EuroSAT](https://github.com/phelber/EuroSAT) | 10 land-use classes (Sentinel-2) | 27,000 images, 64×64 | MIT; `torchvision.datasets.EuroSAT` |
| [GTSRB](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.GTSRB.html) | 43 traffic-sign classes | ~50,000 images | research use; `torchvision.datasets.GTSRB` |
| [MedMNIST v2](https://medmnist.com/) ([PyPI](https://www.pypi.org/project/medmnist)) | 12 biomedical 2D datasets: binary, multi-class, multi-label (ChestMNIST) | 28–224 px | CC BY 4.0 (DermaMNIST: CC BY-NC 4.0); `pip install medmnist` |
| [PatchCamelyon (PCam)](https://github.com/basveeling/pcam) | binary: metastasis in histopathology patches | 327,680 patches, 96×96 | CC0; `torchvision.datasets.PCAM` |

All torchvision built-in datasets: [torchvision.datasets](https://docs.pytorch.org/vision/stable/datasets.html)

---

## Pretrained classification models in `torchvision.models` (ImageNet-1k weights)

### NOte: stick to the ones that does not use 'transformers'. We will do tranformers in coming weeks. This table here is a general guide for pretrained models for computer vision

| Model | Key idea | Params | ImageNet top-1 (approx.) | When to use | Layer to replace |
|---|---|---|---|---|---|
| `resnet18` / `resnet50` | residual (skip) connections | 11.7M / 25.6M | 69.8% / 80.9% | default baseline | `model.fc` |
| `densenet121` | dense connections (concatenation) | 8.0M | 74.4% | fewer parameters than ResNet-50 | `model.classifier` |
| `mobilenet_v3_large` | depthwise separable conv, inverted residuals, squeeze-and-excitation | 5.5M | ~75% | fast; laptops and edge devices | `model.classifier[-1]` |
| `efficientnet_b0` | compound scaling of depth, width, resolution | 5.3M | 77.7% | best accuracy per compute | `model.classifier[-1]` |
| `efficientnet_v2_s` | faster-training EfficientNet | 21.5M | 84.2% | high accuracy, mid-size | `model.classifier[-1]` |
| `regnet_y_*` | architecture from design-space search | various | 78–86% | efficient family across sizes | `model.fc` |
| `convnext_tiny` | CNN modernized with Transformer design choices | 28.6M | 82.5% | strong modern CNN | `model.classifier[-1]` |
| `vit_b_16` | Vision Transformer (no convolutions) | 86.6M | 81.1% | large datasets; contrast with CNNs | `model.heads.head` |
| `swin_t` | Transformer with local windows and a CNN-like hierarchy | 28.3M | 81.5% | Transformer with multi-scale features | `model.head` |
| `vgg16` | plain stacks of 3×3 convolutions | 138M | 71.6% | historical; heavy | `model.classifier[-1]` |

*Accuracies from the torchvision model documentation; they vary slightly with the weight version. Use `weights.transforms()` for each model's input size and normalization (most use 224 px; `efficientnet_v2_s` uses 384).*

**Other sources of pretrained models:** [`timm`](https://github.com/huggingface/pytorch-image-models) (hundreds of models), the [Hugging Face Hub](https://huggingface.co/models), and foundation models such as **CLIP** (zero-shot classification from text labels) and **DINOv2** (strong features for few-label tasks).

---

# Pre-trained models use differnt name for the final layer

Swap the model in notebook 7c: and use below code to use correct name of the final layer:

In [ ]:
from torchvision import models

def load_pretrained(name, num_classes):
    weights = models.get_model_weights(name).DEFAULT
    m = models.get_model(name, weights=weights)
    if hasattr(m, "fc"):                                   # ResNet, RegNet
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif hasattr(m, "heads"):                              # ViT
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    elif hasattr(m, "head"):                               # Swin
        m.head = nn.Linear(m.head.in_features, num_classes)
    elif isinstance(m.classifier, nn.Linear):              # DenseNet
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    else:                                                  # VGG, MobileNet, EfficientNet, ConvNeXt
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    return m, weights.transforms()                         # use the model's own resize + normalization

net, preprocess = load_pretrained("efficientnet_b0", num_classes=10)
print(preprocess)    # e.g. resize 256 -> crop 224, ImageNet mean/std

## Image classification challenges
 (NOte: list created by AI, so do your due diligence if participating)
### Open now (as of October 2026)
| Challenge | Task | Notes |
|---|---|---|
| [DrivenData: Conser-vision Practice Area](https://www.drivendata.org/competitions/87/competition-image-classification-wildlife-conservation/) | classify wildlife species in camera-trap images | practice competition, live leaderboard, no prize; small dataset, open to all levels |
| Kaggle "Getting Started" competitions (e.g., *Digit Recognizer*, *Petals to the Metal*) | handwritten digits; flower species | permanent beginner competitions with rolling leaderboards; check current status on Kaggle |
| [Grand Challenge](https://grand-challenge.org/challenges/) | medical imaging challenges | several open (e.g., OWL until Dec 7, 2026; CLEAR-EC until Nov 15, 2026); use the "Classification" task filter |

### Seasonal (watch for announcements)
| Venue | Typical timing | Examples |
|---|---|---|
| CVPR FGVC workshop ([FGVC12, CVPR 2025](https://cvpr.thecvf.com/virtual/2025/workshop/32343)) | announced early in the year, runs through spring | fine-grained classification competitions hosted on Kaggle |
| LifeCLEF | winter to spring | PlantCLEF, FungiCLEF, AnimalCLEF ([AnimalCLEF26 on Kaggle](https://www.kaggle.com/competitions/animal-clef-2026/overview)) |
| MICCAI challenges | spring to summer | medical image analysis challenges on [Grand Challenge](https://grand-challenge.org/challenges/) |
| [DrivenData](https://www.drivendata.org/competitions/) and Zindi | throughout the year | social-impact prize competitions |